In [84]:
# !pip install -r requirements.txt

In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 파일 불러오기
df_F = pd.read_csv('../../data/raw/sports_gb_F.csv')
df_L = pd.read_csv('../../data/raw/sports_gb_L.csv')
df_total = pd.read_csv('../../data/raw/sports_gb_total.csv')


In [86]:
# 2. 결측치 확인 

# BirthYear가 결측치 확인 / 'BirthYear'에 결측치 존재하므로 제거
df_total[df_total['BirthYear'].isnull()]

df_total = df_total.dropna(subset=['BirthYear'])

# StakeF가 0이 아닌 행들만 골라서 다시 저장
df_F = df_F[df_F['StakeF'] != 0]

# StakeL이 0이 아닌 행들만 골라서 다시 저장
df_L = df_L[df_L['StakeL'] != 0]

In [87]:
# 3. Fixed 베팅 적중률 및 승률 계산

# 적중률 계산
df_F['fixed_hit_rate'] = (df_F['WinF']>0).groupby(df_F['UserID']).transform('mean')

# 승률 계산
df_F['fixed_win_rate'] = (df_F['StakeF'] < df_F['WinF']).groupby(df_F['UserID']).transform('mean')

# 유저 아이디별, 입금일 내림차순 이후 첫행은 유지하고 중복 행 제거 (유저별로 최근 접속일 남기기 위해)
df_F = df_F.sort_values(by=['UserID', 'DateBet'], ascending=False).drop_duplicates(subset=['UserID'], keep='first')

# 적중률, 승률 2자리수 계산
df_F = df_F.round({'fixed_hit_rate': 2, 'fixed_win_rate': 2})
df_F



,UserID,DateBet,StakeF,WinF,BetsF,fixed_hit_rate,fixed_win_rate
2108615,1405190,2006-04-29,6.2100,0.0,7,0.12,0.00
2108605,1405189,2006-07-05,1.0500,0.0,1,0.60,0.28
2108536,1405185,2005-12-04,10.0000,16.0,1,0.50,0.50
2108532,1405184,2006-05-17,5.0000,0.0,1,0.32,0.24
2108495,1405183,2005-05-05,0.5000,0.0,1,0.04,0.04
...,...,...,...,...,...,...,...
459,1324360,2006-03-25,5.5322,0.0,4,0.24,0.14
407,1324358,2005-05-06,63.9172,0.0,1,0.20,0.00
378,1324356,2006-08-25,11.5000,0.0,2,0.25,0.12
315,1324355,2006-08-28,7.0000,0.0,3,0.15,0.14


In [88]:
# 4. Live 베팅 적중률 및 승률 계산

# 적중률 계산
df_L['live_hit_rate'] = (df_L['WinL']>0).groupby(df_L['UserID']).transform('mean')

# 승률 계산
df_L['live_win_rate'] = (df_L['StakeL'] < df_L['WinL']).groupby(df_L['UserID']).transform('mean')

# 유저 아이디별, 입금일 내림차순 이후 첫행은 유지하고 중복 행 제거 (유저별로 최근 접속일 남기기 위해)
df_L = df_L.sort_values(by=['UserID', 'DateBet'], ascending=False).drop_duplicates(subset=['UserID'], keep='first')

# 적중률, 승률 2자리수 계산
df_L = df_L.round({'live_hit_rate': 2, 'live_win_rate': 2})



In [89]:
# 5 적중률 및 승률 df_total에 merge하기

# Fixed 베팅 df_total에 merge 하기
merge1 = pd.merge(df_total,df_F.drop(['StakeF','WinF','BetsF'],axis=1),on='UserID',how='left')

# Live 베팅 df_total에 merge 하기
merge2 = pd.merge(merge1,df_L.drop(['StakeL','WinL','BetsL'],axis=1),on='UserID',how='left')

In [90]:
# 6. 최근 접속일 컬럼 추가 

# 날짜형 변환 (Fixed와 Live 중 최신 베팅 날짜 비교하기 위한)
merge2['DateBet_x'] = pd.to_datetime(merge2['DateBet_x'])
merge2['DateBet_y'] = pd.to_datetime(merge2['DateBet_y'])

# 두 컬럼 중 최신 날짜를 찾아 'latest' 컬럼에 추가
merge2['latest'] = merge2[['DateBet_x', 'DateBet_y']].max(axis=1).astype('datetime64[ns]')

# 'latest' 컬럼만 남기고 이용했던 컬럼 삭제
merge2 = merge2.drop(['DateBet_x','DateBet_y'],axis=1)


In [91]:
# 7. 총 승리 빈도 및 승률 계산

# 결측치 제거
merge2[['fixed_hit_rate', 'live_hit_rate']] = merge2[['fixed_hit_rate', 'live_hit_rate']].fillna(0)
merge2[['fixed_win_rate', 'live_win_rate']] = merge2[['fixed_win_rate', 'live_win_rate']].fillna(0)


# 총 승리 빈도(Total Hit Rate) = 각 타입별 빈도 * 활동 날짜 수 / 총 날짜 수
merge2['total_hit_rate'] = (
    (merge2['fixed_hit_rate'] * merge2['DaysF']) + 
    (merge2['live_hit_rate'] * merge2['DaysL'])
) / merge2['DaysA']

# 소수 2자리 계산
merge2['total_hit_rate'] = merge2['total_hit_rate'].round(2)
merge2

# 총 승률 계산 (Total Win Rate) = 각 타입별 승률 * 활동 날짜 수 / 총 날짜 수
merge2['total_win_rate'] = (
    (merge2['fixed_win_rate'] * merge2['DaysF']) + 
    (merge2['live_win_rate'] * merge2['DaysL'])
) / merge2['DaysA']

# 소수 2자리 계산
merge2['total_win_rate'] = merge2['total_win_rate'].round(2)

In [92]:
# 8. 각 타입별 및 총 수익률 계산 

# fixed 배팅 ROI
merge2['fixed_avg_roi'] = (merge2['WinF'] - merge2['StakeF']) / merge2['StakeF']

# live 배팅 ROI
merge2['live_avg_roi'] = (merge2['WinL'] - merge2['StakeL']) / merge2['StakeL']

# 3. 전체 ROI
merge2['total_avg_roi'] = (merge2['WinA'] - merge2['StakeA']) / merge2['StakeA']

# 결측치 처리 및 소수점 정리
# 라이브 배팅을 안 한 유저 등은 NaN이 생길 수 있으므로 0으로 채우고 반올림
merge2[['fixed_avg_roi', 'live_avg_roi', 'total_avg_roi']] = merge2[['fixed_avg_roi', 'live_avg_roi', 'total_avg_roi']].fillna(0).round(2)

In [93]:
# 9. 관측 날짜로 Churn 계산 


# 기준 날짜 잡기 (관측 마지막 날짜)
merge2['standard'] = pd.to_datetime('2006-08-31')

# 날짜 변환 및 정수형 변환 (불리언 값이므로 0과 1로 나옴) / 휴먼 활동일인 1년 1개월로 임계치(Threshold) 잡기
merge2['Churn'] = ((merge2['standard'] - merge2['latest']).dt.days > 395)

# 이탈률 몇 개인지 확인
merge2['Churn'].value_counts()

Churn
False    33305
True     13032
Name: count, dtype: int64

In [94]:
# 10. 'age_group' 컬럼 만들기 

# standard에서 연도만 뽑아 빼기 / 한국 나이로 계산하므로 1 더하기
merge2['Age'] = (merge2['standard'].dt.year - merge2['BirthYear']) + 1


# cut 함수 사용하여 'age_group' 구하기
bins = [-np.inf, 19, 29, 39, 49, 59, 69, 79, 89, np.inf]
labels = [0, 1, 2, 3, 4, 5, 6, 7, 8]

merge2['age_group'] = pd.cut(merge2['Age'], bins=bins, labels=labels, right=False).astype('Int64').fillna(-1)

In [ ]:
# 11. 컬럼 정리 및 재정의

# 필요 없는 컬럼 정리 
df_final = merge2.drop(['standard','TimeReg','latest','Age','Date1Bet','BirthYear'],axis=1)

# 컬럼 이름 재정의
sports_gb_total = df_final.rename(columns={
    'UserID': 'user_id',
    'CountryID': 'country_id',      
    'Gender': 'gender',             
    'DateReg': 'reg_date',          
    'Date1Dep': 'first_deposit',
    'Date1Bet': 'Date1Bet',         
    'Date1Spo': 'first_bet',
    'StakeF': 'fixed_bet_amount',
    'StakeL': 'live_bet_amount',
    'StakeA': 'total_bet_amount',
    'WinF': 'fixed_win_amount',
    'WinL': 'live_win_amount',
    'WinA': 'total_win_amount',
    'BetsF': 'fixed_bet_cnt',
    'BetsL': 'live_bet_cnt',
    'BetsA': 'total_bet_cnt',
    'DaysF': 'fixed_active_days',
    'DaysL': 'live_active_days',
    'DaysA': 'total_active_days'
})

# 'reg_date', 'first_deposit', 'first_bet' 컬럼을 날짜형으로 변환 
for col in ['reg_date', 'first_deposit', 'first_bet']:
    sports_gb_total[col] = pd.to_datetime(sports_gb_total[col], format='%Y-%m-%d')

# 'gender'컬럼을 bool값으로 변환
sports_gb_total['gender'] = sports_gb_total['gender'].astype(bool)

In [96]:
# 12. 파일저장
sports_gb_total.to_csv('sports_gb_total_final.csv', index=False, encoding='utf-8-sig')

In [97]:
sports_gb_total

,user_id,country_id,gender,reg_date,first_deposit,first_bet,fixed_bet_amount,live_bet_amount,total_bet_amount,fixed_win_amount,...,fixed_win_rate,live_hit_rate,live_win_rate,total_hit_rate,total_win_rate,fixed_avg_roi,live_avg_roi,total_avg_roi,Churn,age_group
0,1324354,276,True,2005-02-01,2005-02-24,2005-02-24,15750.3800,2146.4700,17896.8500,15010.9000,...,0.24,0.67,0.42,0.49,0.30,-0.05,-0.16,-0.06,False,3
1,1324355,300,True,2005-02-01,2005-02-01,2005-02-01,639.2998,24.7000,663.9998,569.3700,...,0.14,0.14,0.14,0.16,0.15,-0.11,-0.55,-0.13,False,1
2,1324356,276,True,2005-02-01,2005-02-01,2005-02-02,898.8100,701.8200,1600.6300,336.3600,...,0.12,0.67,0.26,0.56,0.24,-0.63,-0.07,-0.38,False,2
3,1324358,752,True,2005-02-01,2005-02-01,2005-02-01,247.6970,88.5927,336.2897,153.8755,...,0.00,1.00,0.00,0.40,0.00,-0.38,-0.37,-0.38,True,1
4,1324360,792,True,2005-02-01,2005-02-02,2005-02-02,685.9424,6.6641,692.6065,623.8984,...,0.14,0.50,0.25,0.27,0.15,-0.09,-0.54,-0.09,False,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46332,1405183,276,True,2005-02-28,2005-03-03,2005-03-04,88.7600,102.1300,190.8900,21.4600,...,0.04,0.62,0.23,0.37,0.16,-0.76,-0.17,-0.45,True,3
46333,1405184,276,True,2005-02-28,2005-02-28,2005-03-01,315.8800,296.7500,612.6300,226.8600,...,0.24,0.65,0.39,0.72,0.48,-0.28,0.08,-0.11,False,2
46334,1405185,276,True,2005-02-28,2005-11-08,2005-11-08,13.0000,124.7100,137.7100,16.0000,...,0.50,0.40,0.20,0.60,0.40,0.23,-0.30,-0.25,False,2
46335,1405189,276,True,2005-02-28,2005-03-01,2005-03-01,414.0900,10.0000,424.0900,421.2900,...,0.28,0.67,0.67,0.63,0.31,0.02,0.78,0.04,False,1


In [98]:
sports_gb_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46337 entries, 0 to 46336
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   user_id            46337 non-null  int64         
 1   country_id         46337 non-null  int64         
 2   gender             46337 non-null  bool          
 3   reg_date           46337 non-null  datetime64[ns]
 4   first_deposit      46337 non-null  datetime64[ns]
 5   first_bet          46337 non-null  datetime64[ns]
 6   fixed_bet_amount   46337 non-null  float64       
 7   live_bet_amount    46337 non-null  float64       
 8   total_bet_amount   46337 non-null  float64       
 9   fixed_win_amount   46337 non-null  float64       
 10  live_win_amount    46337 non-null  float64       
 11  total_win_amount   46337 non-null  float64       
 12  fixed_bet_cnt      46337 non-null  int64         
 13  live_bet_cnt       46337 non-null  int64         
 14  total_